In [ ]:
!pip install safe-ai-metrics

In [4]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# SAFE-AI metrics from the package
from safeai.rga import compare_models_rga_binary
from safeai.rgr import compare_models_rgr        # you can try also compare_models_rgr_adversarial, check usage in package code
from safeai.rge import compare_models_rge_tabular


# Simple binary classification dataset
x, y = make_classification(
    n_samples=800,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_repeated=0,
    n_classes=2,
    weights=[0.75, 0.25],
    class_sep=1.0,
    random_state=42
)

feature_names = np.array([
    'Feature_1',
    'Feature_2',
    'Feature_3',
    'Feature_4',
    'Feature_5',
    'Feature_6'
])

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print('Train shape:', x_train.shape)
print('Test shape: ', x_test.shape)
print('Class distribution in test:', np.bincount(y_test))


# Train 2 models
logistic_model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

models = {
    'Logistic Regression': logistic_model,
    'Random Forest': random_forest_model
}

for name, model in models.items():
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    y_prob = model.predict_proba(x_test)[:, 1]

    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)
    print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))
    print(classification_report(y_test, y_pred))



# Probabilities
prob_logistic = logistic_model.predict_proba(x_test)
prob_rf = random_forest_model.predict_proba(x_test)

prob_dict_binary = {
    'Logistic Regression': prob_logistic[:, 1],
    'Random Forest': prob_rf[:, 1]
}

prob_dict_full = {
    'Logistic Regression': prob_logistic,
    'Random Forest': prob_rf
}

class_order = np.array([0, 1])

# RGA
print('\n' + '=' * 70)
print('RGA COMPARISON')
print('=' * 70)

rga_results = compare_models_rga_binary(
    models_dict=prob_dict_binary,
    y=y_test,
    positive_class=1,
    n_segments=10,
    verbose=True,
    save_path='demo_rga_binary.png'
)

rga_dict = {
    model_name: result['full_rga']
    for model_name, result in rga_results.items()
}

print('\nRGA summary:')
for model_name, rga_value in rga_dict.items():
    print(f'{model_name:20s}: RGA = {rga_value:.4f}')


# RGR
print('\n' + '=' * 70)
print('RGR COMPARISON')
print('=' * 70)

noise_levels = np.linspace(0.0, 0.5, 6) # here level of gaussian noise, so you can choose another values

rgr_models_dict = {
    'Logistic Regression': (
        logistic_model,
        x_test,
        prob_logistic,
        logistic_model.classes_,
        'sklearn',
        None
    ),
    'Random Forest': (
        random_forest_model,
        x_test,
        prob_rf,
        random_forest_model.classes_,
        'sklearn',
        None
    )
}

rgr_results = compare_models_rgr(
    models_dict=rgr_models_dict,
    noise_levels=noise_levels,
    class_order=class_order,
    rga_dict=rga_dict,
    verbose=True,
    random_seed=42,
    save_path='demo_rgr_binary.png'
)

print('\nRGR summary:')
for model_name, result in rgr_results.items():
    print(f'{model_name:20s}: AURGR = {result["aurgr"]:.4f}')


# RGE
print('\n' + '=' * 70)
print('RGE COMPARISON')
print('=' * 70)

rge_models_dict = {
    'Logistic Regression': (
        logistic_model,
        x_test,
        feature_names,
        prob_logistic,
        logistic_model.classes_,
        'sklearn',
        None
    ),
    'Random Forest': (
        random_forest_model,
        x_test,
        feature_names,
        prob_rf,
        random_forest_model.classes_,
        'sklearn',
        None
    )
}

rge_results = compare_models_rge_tabular(
    models_dict=rge_models_dict,
    class_order=class_order,
    rga_dict=rga_dict,
    masking_method='greedy',
    baseline='mean',
    n_steps=len(feature_names),
    verbose=True,
    random_seed=42,
    save_path='demo_rge_binary.png'
)

print('\nRGE summary:')
for model_name, result in rge_results.items():
    print(f'\n{model_name}')
    print(f'AURGE = {result["aurge"]:.4f}')
    print('Feature removal order:', result['removed_features'])



# Final small summary
print('\n' + '=' * 70)
print('FINAL SAFE-AI DEMO SUMMARY')
print('=' * 70)

for model_name in models.keys():
    print(f'\n{model_name}')
    print(f'RGA   = {rga_results[model_name]['full_rga']:.4f}')
    print(f'AURGR = {rgr_results[model_name]['aurgr']:.4f}')
    print(f'AURGE = {rge_results[model_name]['aurge']:.4f}')

print('\nPlots saved as:')
print(' - demo_rga_binary.png')
print(' - demo_rgr_binary.png')
print(' - demo_rge_binary.png')


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\koles\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


Train shape: (560, 6)
Test shape:  (240, 6)
Class distribution in test: [179  61]

Logistic Regression
ROC-AUC: 0.7454
              precision    recall  f1-score   support

           0       0.87      0.77      0.82       179
           1       0.50      0.67      0.57        61

    accuracy                           0.75       240
   macro avg       0.69      0.72      0.70       240
weighted avg       0.78      0.75      0.76       240


Random Forest
ROC-AUC: 0.9068
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       179
           1       0.80      0.61      0.69        61

    accuracy                           0.86       240
   macro avg       0.84      0.78      0.80       240
weighted avg       0.86      0.86      0.86       240


RGA COMPARISON

Evaluating Logistic Regression...
Logistic Regression: RGA=0.7247

Evaluating Random Forest...
Random Forest: RGA=0.8937

RGA summary:
Logistic Regression : RGA = 0.7247
Random Fo